# Customer Churn Prediction & Retention Analysis
### Veda Technology | AI & ML Internship

**Objective:** Predict whether a telecom customer may leave, analyze churn patterns, compare classification models, and suggest retention actions.

**Dataset:** IBM Telco Customer Churn public sample dataset.

In [ ]:
%pip -q install pandas numpy matplotlib seaborn scikit-learn joblib

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report, RocCurveDisplay
from sklearn.inspection import permutation_importance
import joblib
RANDOM_STATE=42
pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")
print("Libraries imported successfully")

## 1. Load the dataset
The CSV is downloaded from the public IBM GitHub repository. Internet access is required.

In [ ]:
DATA_URL="https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
df=pd.read_csv(DATA_URL)
print("Dataset shape:", df.shape)
display(df.head())

## 2. Data inspection and cleaning
`customerID` is an identifier, not a predictive feature. Convert `TotalCharges` to numeric; blank values become missing and are imputed later in the pipeline.

In [ ]:
df.info()
print("Missing values before cleaning:")
display(df.isna().sum().sort_values(ascending=False).head(10))
print("Duplicate rows:", df.duplicated().sum())
df=df.drop_duplicates().drop_duplicates(subset=["customerID"]).copy()
df["TotalCharges"]=pd.to_numeric(df["TotalCharges"], errors="coerce")
df["Churn"]=df["Churn"].astype(str).str.strip()
df=df[df["Churn"].isin(["Yes","No"])].copy()
print("Cleaned shape:", df.shape)
print("Missing TotalCharges:", df["TotalCharges"].isna().sum())

## 3. Exploratory Data Analysis
These visualizations describe patterns in this sample; they do not prove causation.

In [ ]:
plt.figure(figsize=(6,4)); sns.countplot(data=df,x="Churn"); plt.title("Customer Churn Distribution"); plt.tight_layout(); plt.show()
print(f"Churn rate: {df.Churn.eq('Yes').mean()*100:.2f}%")

In [ ]:
plt.figure(figsize=(8,4)); sns.countplot(data=df,x="Contract",hue="Churn"); plt.title("Churn by Contract Type"); plt.xticks(rotation=10); plt.tight_layout(); plt.show()

In [ ]:
plt.figure(figsize=(8,4)); sns.histplot(data=df,x="tenure",hue="Churn",bins=25,multiple="stack"); plt.title("Tenure Distribution by Churn"); plt.tight_layout(); plt.show()

In [ ]:
plt.figure(figsize=(8,4)); sns.boxplot(data=df,x="Churn",y="MonthlyCharges"); plt.title("Monthly Charges by Churn"); plt.tight_layout(); plt.show()

In [ ]:
plt.figure(figsize=(8,4)); sns.countplot(data=df,x="InternetService",hue="Churn"); plt.title("Churn by Internet Service"); plt.tight_layout(); plt.show()

## 4. Feature engineering and target selection
Create `ServiceCount` from selected service columns. The target is `Churn` (No=0, Yes=1).

In [ ]:
service_cols=["PhoneService","MultipleLines","OnlineSecurity","OnlineBackup","DeviceProtection","TechSupport","StreamingTV","StreamingMovies"]
df["ServiceCount"]=df[service_cols].eq("Yes").sum(axis=1)
X=df.drop(columns=["customerID","Churn"])
y=df["Churn"].map({"No":0,"Yes":1}).astype(int)
print("X shape:",X.shape," y shape:",y.shape)
print("Target counts:"); print(y.value_counts())

## 5. Train/test split and preprocessing
Use stratified splitting. Numeric columns are median-imputed and scaled; categorical columns are mode-imputed and one-hot encoded. The pipeline fits preprocessing only on training data.

In [ ]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.20,random_state=RANDOM_STATE,stratify=y)
numeric_features=X.select_dtypes(include=["int64","float64"]).columns.tolist()
categorical_features=X.select_dtypes(exclude=["int64","float64"]).columns.tolist()
numeric_transformer=Pipeline([("imputer",SimpleImputer(strategy="median")),("scaler",StandardScaler())])
categorical_transformer=Pipeline([("imputer",SimpleImputer(strategy="most_frequent")),("onehot",OneHotEncoder(handle_unknown="ignore"))])
preprocessor=ColumnTransformer([("num",numeric_transformer,numeric_features),("cat",categorical_transformer,categorical_features)])
print("Train rows:",len(X_train)," Test rows:",len(X_test))

## 6. Train three classification models
Compare Logistic Regression, Decision Tree, and Random Forest. Class weights help account for class imbalance.

In [ ]:
models={
 "Logistic Regression":LogisticRegression(max_iter=2000,class_weight="balanced",random_state=RANDOM_STATE),
 "Decision Tree":DecisionTreeClassifier(max_depth=6,min_samples_leaf=10,class_weight="balanced",random_state=RANDOM_STATE),
 "Random Forest":RandomForestClassifier(n_estimators=250,min_samples_leaf=3,class_weight="balanced",random_state=RANDOM_STATE,n_jobs=-1)
}
pipelines={}; results=[]
for name,estimator in models.items():
    pipe=Pipeline([("preprocessor",preprocessor),("model",estimator)])
    pipe.fit(X_train,y_train); pipelines[name]=pipe
    pred=pipe.predict(X_test); prob=pipe.predict_proba(X_test)[:,1]
    results.append({"Model":name,"Accuracy":accuracy_score(y_test,pred),"Precision":precision_score(y_test,pred,zero_division=0),"Recall":recall_score(y_test,pred,zero_division=0),"F1-score":f1_score(y_test,pred,zero_division=0),"ROC-AUC":roc_auc_score(y_test,prob)})
comparison=pd.DataFrame(results).sort_values("ROC-AUC",ascending=False)
display(comparison.round(3))

## 7. ROC curves and model selection
Accuracy alone can be misleading for imbalanced data. Review precision, recall, F1-score, and ROC-AUC together.

In [ ]:
fig,ax=plt.subplots(figsize=(8,6))
for name,pipe in pipelines.items(): RocCurveDisplay.from_estimator(pipe,X_test,y_test,name=name,ax=ax)
plt.title("ROC Curves - Model Comparison"); plt.tight_layout(); plt.show()
best_model_name=comparison.iloc[0]["Model"]
best_model=pipelines[best_model_name]
print("Best model by test ROC-AUC:",best_model_name)

## 8. Confusion matrix and classification report

In [ ]:
best_predictions=best_model.predict(X_test)
cm=confusion_matrix(y_test,best_predictions)
plt.figure(figsize=(5,4)); sns.heatmap(cm,annot=True,fmt="d",cmap="Blues",xticklabels=["Predicted No","Predicted Yes"],yticklabels=["Actual No","Actual Yes"]); plt.title(f"Confusion Matrix - {best_model_name}"); plt.tight_layout(); plt.show()
print(classification_report(y_test,best_predictions,target_names=["No Churn","Churn"],zero_division=0))

## 9. Feature importance
Permutation importance estimates how much ROC-AUC falls when one feature is shuffled. It is an association/importance measure, not proof of causation.

In [ ]:
perm=permutation_importance(best_model,X_test,y_test,n_repeats=5,random_state=RANDOM_STATE,scoring="roc_auc",n_jobs=-1)
importance=pd.DataFrame({"Feature":X_test.columns,"Importance":perm.importances_mean}).sort_values("Importance",ascending=False)
display(importance.head(15).round(4))
top=importance.head(12).sort_values("Importance")
plt.figure(figsize=(8,6)); plt.barh(top.Feature,top.Importance); plt.title("Top Feature Importance"); plt.xlabel("Mean decrease in ROC-AUC when shuffled"); plt.tight_layout(); plt.show()

## 10. Customer churn risk prediction function
Provide a customer record containing the original input feature fields (excluding customerID and Churn). The function calculates `ServiceCount` and returns the churn probability and a simple risk band.

In [ ]:
def predict_customer_churn(customer_record, model=best_model):
    customer_df=pd.DataFrame([customer_record])
    customer_df["ServiceCount"]=customer_df[service_cols].eq("Yes").sum(axis=1)
    customer_df=customer_df.reindex(columns=X.columns)
    probability=float(model.predict_proba(customer_df)[0,1])
    predicted="Yes" if probability>=0.50 else "No"
    risk="High" if probability>=0.70 else ("Medium" if probability>=0.40 else "Low")
    return {"churn_probability":round(probability,4),"predicted_churn":predicted,"risk_level":risk}

# Example using one held-out customer record
sample_customer=X_test.iloc[0].drop(labels=["ServiceCount"]).to_dict()
print(predict_customer_churn(sample_customer))

## 11. Retention recommendations
Use the observed patterns and model explanations to develop hypotheses. Validate actions with business teams and controlled experiments.

In [ ]:
recommendations=[
 "Review high-risk customers with short tenure and offer onboarding support.",
 "Consider suitable contract options for month-to-month customers.",
 "Review billing clarity and payment friction for customers with high charges.",
 "Offer relevant technical support and service-quality checks.",
 "Prioritize outreach using churn probability, customer value, and contact preferences."
]
for i,item in enumerate(recommendations,1): print(f"{i}. {item}")
print("\nThese are suggestions to test, not proof that an offer will prevent churn.")

## 12. Save the trained model pipeline

In [ ]:
joblib.dump({"model":best_model,"feature_columns":X.columns.tolist(),"service_columns":service_cols,"model_name":best_model_name},"customer_churn_model.joblib")
print("Saved customer_churn_model.joblib")

## 13. Conclusion
This project demonstrates data cleaning, EDA, feature engineering, preprocessing, three classification models, model evaluation, feature importance, churn-risk prediction, and retention recommendations.

**Limitations:** This is a cross-sectional educational sample. Scores depend on the train/test split; the model is not a guaranteed future forecast, and observed associations do not establish causation.